In [ ]:
import pandas as pd
from collections import Counter
import re
import os
# Load dataset
df = pd.read_csv("sample.csv")

# ✅ Combine 'findings' and 'impression' into one report
df['report'] = df['findings'].fillna('') + ' ' + df['impression'].fillna('')

# ✅ Preprocess: lowercase, remove punctuation, tokenize
df["report_clean"] = df["report"].str.lower().apply(lambda x: re.sub(r"[^a-z\s]", "", x)).str.split()

# ✅ Build vocabulary with minimum frequency threshold (e.g., 5)
all_words = [word for report in df["report_clean"] for word in report]
word_counts = Counter(all_words)
vocab = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"] + [w for w, c in word_counts.items() if c >= 5]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

# ✅ Encode reports into sequences of indices
def encode_report(report):
    return [word2idx.get("<SOS>")] + \
           [word2idx.get(word, word2idx["<UNK>"]) for word in report] + \
           [word2idx.get("<EOS>")]

df["encoded"] = df["report_clean"].apply(encode_report)

# ✅ (Optional) Save to CSV
df.to_csv("encoded_dataset.csv", index=False)

# ✅ Show sample output
print("✅ Done. Sample:")
print(df[["findings", "impression", "report", "encoded"]].head())
print(f"📚 Vocabulary size: {len(vocab)}")


✅ Sample:
                                            findings  \
0  the cardiac silhouette and mediastinum size ar...   
1  the cardiac silhouette and mediastinum size ar...   
2  borderline cardiomegaly midline sternotomy enl...   
3  borderline cardiomegaly midline sternotomy enl...   
4                                     no information   

                                          impression  \
0                                       normal chest   
1                                       normal chest   
2                        no acute pulmonary findings   
3                        no acute pulmonary findings   
4  no displaced rib fractures pneumothorax or ple...   

                                              report  \
0  the cardiac silhouette and mediastinum size ar...   
1  the cardiac silhouette and mediastinum size ar...   
2  borderline cardiomegaly midline sternotomy enl...   
3  borderline cardiomegaly midline sternotomy enl...   
4  no information no displaced rib f

In [ ]:
from sklearn.model_selection import train_test_split

# First shuffle and split train vs temp (temp will later be split into valid/test)
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, shuffle=True)

# Split temp into validation and test (each 10%)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, shuffle=True)

# Print sizes
print(f"Train size: {len(train_df)}")
print(f"Valid size: {len(valid_df)}")
print(f"Test  size: {len(test_df)}")


Train: 799 | Valid: 100 | Test: 100
📐 Shape of padded sequences: torch.Size([999, 220])


In [4]:
import torch
from torch.nn.utils.rnn import pad_sequence

# Convert list of encoded sequences to tensors
encoded_sequences = [torch.tensor(seq, dtype=torch.long) for seq in df['encoded']]

# Pad sequences to the same length
padded_sequences = pad_sequence(encoded_sequences, batch_first=True, padding_value=word2idx["<PAD>"])

# Result: padded_sequences is a [N, max_len] tensor
print("Shape of padded sequences:", padded_sequences.shape)


Shape of padded sequences: torch.Size([999, 220])


In [5]:
from torch.utils.data import Dataset
from PIL import Image
from torchvision import transforms

class CXRRNNDataset(Dataset):
    def __init__(self, df, word2idx, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.word2idx = word2idx
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # Load and transform the image
        image_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)

        # Get encoded report
        report = torch.tensor(row['encoded'], dtype=torch.long)
        return image, report


In [8]:
# Define validation dataset
valid_dataset = CXRRNNDataset(valid_df, word2idx, image_dir="C:/Users/ag331/OneDrive/Desktop/med_project/archive/_Images/_Images_normalized", transform=image_transforms)


In [7]:
image_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# Example usage:
train_dataset = CXRRNNDataset(train_df, word2idx, image_dir="C:/Users/ag331/OneDrive/Desktop/med_project/archive/_Images/_Images_normalized", transform=image_transforms)


In [9]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    images, reports = zip(*batch)
    images = torch.stack(images)
    reports = [torch.tensor(r, dtype=torch.long) for r in reports]
    padded_reports = pad_sequence(reports, batch_first=True, padding_value=word2idx["<PAD>"])
    return images, padded_reports

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)


In [10]:
import torch
import torch.nn as nn
import torchvision.models as models

class CNN_Encoder(nn.Module):
    def __init__(self, encoded_dim=512):
        super(CNN_Encoder, self).__init__()
        resnet = models.resnet50(pretrained=True)
        self.resnet = nn.Sequential(*list(resnet.children())[:-1])  # Remove FC
        self.linear = nn.Linear(resnet.fc.in_features, encoded_dim)
        self.bn = nn.BatchNorm1d(encoded_dim)

    def forward(self, images):
        features = self.resnet(images).squeeze()
        features = self.linear(features)
        return self.bn(features)

class LSTM_Decoder(nn.Module):
    def __init__(self, encoded_dim, hidden_dim, vocab_size):
        super(LSTM_Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.init_h = nn.Linear(encoded_dim, hidden_dim)
        self.init_c = nn.Linear(encoded_dim, hidden_dim)

    def forward(self, features, captions):
        embeddings = self.embedding(captions)
        h0 = self.init_h(features).unsqueeze(0)
        c0 = self.init_c(features).unsqueeze(0)
        outputs, _ = self.lstm(embeddings, (h0, c0))
        return self.fc(outputs)


In [11]:
# Set up
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab_size = len(word2idx)
encoder = CNN_Encoder().to(device)
decoder = LSTM_Decoder(encoded_dim=512, hidden_dim=512, vocab_size=vocab_size).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word2idx["<PAD>"])
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-4)

# Training loop
for epoch in range(30):
    encoder.train()
    decoder.train()
    total_loss = 0

    for images, reports in train_loader:
        images, reports = images.to(device), reports.to(device)

        optimizer.zero_grad()
        features = encoder(images)  # [B, 512]
        input_seq = reports[:, :-1]
        target_seq = reports[:, 1:]

        outputs = decoder(features, input_seq)  # [B, T, vocab]
        loss = criterion(outputs.view(-1, vocab_size), target_seq.reshape(-1))

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")


c:\Users\ag331\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\ag331\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
C:\Users\ag331\AppData\Local\Temp\ipykernel_74052\318688588.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  reports = [torch.tensor(r, dtype=torch.long) fo

Epoch 1, Loss: 6.0819
Epoch 2, Loss: 5.4471
Epoch 3, Loss: 4.4463
Epoch 4, Loss: 3.9901
Epoch 5, Loss: 3.7076
Epoch 6, Loss: 3.4959
Epoch 7, Loss: 3.3137
Epoch 8, Loss: 3.1597
Epoch 9, Loss: 3.0163
Epoch 10, Loss: 2.8996
Epoch 11, Loss: 2.7941
Epoch 12, Loss: 2.6913
Epoch 13, Loss: 2.5869
Epoch 14, Loss: 2.4967
Epoch 15, Loss: 2.4152
Epoch 16, Loss: 2.3205
Epoch 17, Loss: 2.2469
Epoch 18, Loss: 2.1650
Epoch 19, Loss: 2.0904
Epoch 20, Loss: 2.0211
Epoch 21, Loss: 1.9544
Epoch 22, Loss: 1.8834
Epoch 23, Loss: 1.8264
Epoch 24, Loss: 1.7584
Epoch 25, Loss: 1.6945
Epoch 26, Loss: 1.6436
Epoch 27, Loss: 1.5906
Epoch 28, Loss: 1.5346
Epoch 29, Loss: 1.4857
Epoch 30, Loss: 1.4332


In [20]:
def generate_report_beam(image, encoder, decoder, word2idx, idx2word, beam_width=5, max_len=100):
    encoder.eval()
    decoder.eval()

    with torch.no_grad():
        # Sanity check
        if image.dim() == 1:
            raise ValueError(f"Invalid image shape: {image.shape}. Expected at least 3D.")

        # Add batch dimension
        image = image.unsqueeze(0).to(device)  # shape: [1, 3, 224, 224]
        
        # Extract image features
        features = encoder(image)  # shape: [1, 512]

        # Beam search initialization
        sequences = [[word2idx["<SOS>"]]]
        scores = torch.zeros(1, device=device)

        for _ in range(max_len):
            all_candidates = []
            for i, seq in enumerate(sequences):
                if seq[-1] == word2idx["<EOS>"]:
                    all_candidates.append((seq, scores[i]))
                    continue

                input_seq = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)
                embedded = decoder.embedding(input_seq)
                h0 = decoder.init_h(features).unsqueeze(0)
                c0 = decoder.init_c(features).unsqueeze(0)
                output, _ = decoder.lstm(embedded, (h0, c0))
                logits = decoder.fc(output[:, -1, :])
                log_probs = torch.log_softmax(logits, dim=-1)
                topk_log_probs, topk_idx = torch.topk(log_probs, beam_width)

                for j in range(beam_width):
                    new_seq = seq + [topk_idx[0, j].item()]
                    new_score = scores[i] + topk_log_probs[0, j]
                    all_candidates.append((new_seq, new_score))

            ordered = sorted(all_candidates, key=lambda x: x[1] / len(x[0]), reverse=True)
            sequences, scores = zip(*ordered[:beam_width])
            scores = torch.tensor(scores)

            if all(seq[-1] == word2idx["<EOS>"] for seq in sequences):
                break

        best_seq = sequences[0]
        return " ".join(idx2word[idx] for idx in best_seq if idx not in [word2idx["<PAD>"], word2idx["<SOS>"], word2idx["<EOS>"]])


In [21]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge

def evaluate_bleu_rouge(gts, preds):
    smoothie = SmoothingFunction().method1
    bleu_scores = []
    rouge = Rouge()
    rouge_scores = []

    for ref, hyp in zip(gts, preds):
        ref = ref.lower().split()
        hyp = hyp.lower().split()
        bleu = sentence_bleu([ref], hyp, smoothing_function=smoothie)
        bleu_scores.append(bleu)
        rouge_scores.append(rouge.get_scores(" ".join(hyp), " ".join(ref))[0])

    avg_bleu = sum(bleu_scores) / len(bleu_scores)
    avg_rouge = {
        metric: sum([s[metric]["f"] for s in rouge_scores]) / len(rouge_scores)
        for metric in ["rouge-1", "rouge-2", "rouge-l"]
    }

    return avg_bleu, avg_rouge


In [22]:
def decode_report(indices):
    return " ".join(idx2word.get(i, "<UNK>") for i in indices if i not in [word2idx["<PAD>"]])


In [10]:
# Generate reports from the validation dataset
# Generate reports from the validation dataset
ground_truth_reports = []
generated_reports = []

for i in range(len(valid_dataset)):
    image, report_tensor = valid_dataset[i]
    
    if image.dim() != 3:
        print(f"⚠️ Image at index {i} has shape {image.shape} — skipping")
        continue
    
    ground_truth = decode_report(report_tensor.tolist())
    generated = generate_report_beam(image, encoder, decoder, word2idx, idx2word)

    ground_truth_reports.append(ground_truth)
    generated_reports.append(generated)



In [16]:
# Evaluation
bleu, rouge = evaluate_bleu_rouge(ground_truth_reports, generated_reports)

# Optional: if you implemented CheXbert F1
# f1 = compute_f1_chexbert(ground_truth_reports, generated_reports)

print("BLEU:", bleu)
print("ROUGE:", rouge)
# print("F1 CheXbert:", f1)


ZeroDivisionError: division by zero

In [20]:
import os

image_dir = "C:/Users/ag331/OneDrive/Desktop/med_project/archive/_Images/_Images_normalized"

# Remove rows with missing image files
df = df[df["filename"].apply(lambda x: os.path.isfile(os.path.join(image_dir, x)))].copy()
print("✅ Verified files. Remaining:", len(df))


✅ Verified files. Remaining: 0


In [21]:
print(df["filename"].head())


Series([], Name: filename, dtype: object)


In [22]:
import os
print("Files in directory:")
print(os.listdir(image_dir)[:10])


Files in directory:
['1000_IM-0003-1001.dcm_Frontal.png', '1000_IM-0003-2001.dcm_Lateral.png', '1000_IM-0003-3001.dcm.png', '1001_IM-0004-1001.dcm_Frontal.png', '1001_IM-0004-1002.dcm_Lateral.png', '1002_IM-0004-1001.dcm_Frontal.png', '1002_IM-0004-2001.dcm_Lateral.png', '1003_IM-0005-2002.dcm_Frontal.png', '1004_IM-0005-1001.dcm_Frontal.png', '1004_IM-0005-2001.dcm_Lateral.png']


In [3]:
# =====================
# 1. PREPROCESSING
# =====================
import pandas as pd
from collections import Counter
import re
import os

df = pd.read_csv("sample.csv")
df['report'] = df['findings'].fillna('') + ' ' + df['impression'].fillna('')
df["report_clean"] = df["report"].str.lower().apply(lambda x: re.sub(r"[^a-z\s]", "", x)).str.split()

all_words = [word for report in df["report_clean"] for word in report]
word_counts = Counter(all_words)
vocab = ["<PAD>", "<SOS>", "<EOS>", "<UNK>"] + [w for w, c in word_counts.items() if c >= 5]
word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for w, i in word2idx.items()}

def encode_report(report):
    return [word2idx["<SOS>"]] + [word2idx.get(word, word2idx["<UNK>"]) for word in report] + [word2idx["<EOS>"]]

df["encoded"] = df["report_clean"].apply(encode_report)
df.to_csv("encoded_dataset.csv", index=False)

print("✅ Sample:")
print(df[["findings", "impression", "report", "encoded"]].head())
print(f"📚 Vocabulary size: {len(vocab)}")


✅ Sample:
                                            findings  \
0  the cardiac silhouette and mediastinum size ar...   
1  the cardiac silhouette and mediastinum size ar...   
2  borderline cardiomegaly midline sternotomy enl...   
3  borderline cardiomegaly midline sternotomy enl...   
4                                     no information   

                                          impression  \
0                                       normal chest   
1                                       normal chest   
2                        no acute pulmonary findings   
3                        no acute pulmonary findings   
4  no displaced rib fractures pneumothorax or ple...   

                                              report  \
0  the cardiac silhouette and mediastinum size ar...   
1  the cardiac silhouette and mediastinum size ar...   
2  borderline cardiomegaly midline sternotomy enl...   
3  borderline cardiomegaly midline sternotomy enl...   
4  no information no displaced rib f

In [4]:
# =====================
# 2. SPLITTING & PADDING
# =====================
from sklearn.model_selection import train_test_split
import torch
from torch.nn.utils.rnn import pad_sequence

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42)
valid_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

print(f"Train: {len(train_df)} | Valid: {len(valid_df)} | Test: {len(test_df)}")

encoded_sequences = [torch.tensor(seq, dtype=torch.long) for seq in df['encoded']]
padded_sequences = pad_sequence(encoded_sequences, batch_first=True, padding_value=word2idx["<PAD>"])
print("📐 Shape of padded sequences:", padded_sequences.shape)


Train: 799 | Valid: 100 | Test: 100
📐 Shape of padded sequences: torch.Size([999, 220])


In [5]:
# =====================
# 3. DATASET & DATALOADER
# =====================
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from torchvision import transforms

image_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class CXRRNNDataset(Dataset):
    def __init__(self, df, word2idx, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = os.path.join(self.image_dir, row["filename"])
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        report = torch.tensor(row['encoded'], dtype=torch.long)
        return image, report

def collate_fn(batch):
    images, reports = zip(*batch)
    images = torch.stack(images)
    padded_reports = pad_sequence(reports, batch_first=True, padding_value=word2idx["<PAD>"])
    return images, padded_reports

image_dir = "C:/Users/ag331/OneDrive/Desktop/med_project/archive/_Images/_Images_normalized"
train_dataset = CXRRNNDataset(train_df, word2idx, image_dir, image_transforms)
valid_dataset = CXRRNNDataset(valid_df, word2idx, image_dir, image_transforms)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)


In [6]:
# =====================
# 4. CNN + LSTM MODEL
# =====================
import torch.nn as nn
import torchvision.models as models

class CNN_Encoder(nn.Module):
    def __init__(self, encoded_dim=512):
        super(CNN_Encoder, self).__init__()
        resnet = models.resnet50(pretrained=True)
        self.resnet = nn.Sequential(*list(resnet.children())[:-1])
        self.linear = nn.Linear(resnet.fc.in_features, encoded_dim)
        self.bn = nn.BatchNorm1d(encoded_dim)

    def forward(self, images):
        features = self.resnet(images)  # [B, 2048, 1, 1]
        features = features.view(features.size(0), -1)  # [B, 2048]
        features = self.linear(features)
        return self.bn(features)  # [B, 512]

class LSTM_Decoder(nn.Module):
    def __init__(self, encoded_dim, hidden_dim, vocab_size):
        super(LSTM_Decoder, self).__init__()
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.init_h = nn.Linear(encoded_dim, hidden_dim)
        self.init_c = nn.Linear(encoded_dim, hidden_dim)

    def forward(self, features, captions):
        embeddings = self.embedding(captions)
        h0 = self.init_h(features).unsqueeze(0)
        c0 = self.init_c(features).unsqueeze(0)
        outputs, _ = self.lstm(embeddings, (h0, c0))
        return self.fc(outputs)


In [ ]:
# =====================
# 5. TRAINING LOOP
# =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab_size = len(word2idx)

encoder = CNN_Encoder().to(device)
decoder = LSTM_Decoder(encoded_dim=512, hidden_dim=512, vocab_size=vocab_size).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word2idx["<PAD>"])
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=1e-4)

for epoch in range(100):
    encoder.train()
    decoder.train()
    total_loss = 0
    for images, reports in train_loader:
        images, reports = images.to(device), reports.to(device)
        optimizer.zero_grad()
        features = encoder(images)
        outputs = decoder(features, reports[:, :-1])
        loss = criterion(outputs.view(-1, vocab_size), reports[:, 1:].reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_loader):.4f}")


✅ Epoch 1 | Loss: 5.7339 | Time: 133.4s
🔸 Best model saved.
                                                                     
✅ Epoch 2 | Loss: 4.8659 | Time: 130.8s
🔸 Best model saved.
                                                                     
✅ Epoch 3 | Loss: 4.6152 | Time: 128.3s
🔸 Best model saved.
                                                                     
✅ Epoch 4 | Loss: 4.4022 | Time: 130.1s
🔸 Best model saved.
                                                                     
✅ Epoch 5 | Loss: 4.1661 | Time: 128.0s
🔸 Best model saved.
                                                                     
✅ Epoch 6 | Loss: 3.9552 | Time: 128.9s
🔸 Best model saved.
                                                                     
✅ Epoch 7 | Loss: 3.7595 | Time: 128.2s
🔸 Best model saved.
                                                                     
✅ Epoch 8 | Loss: 3.5900 | Time: 128.2s
🔸 Best model saved.
                              

In [ ]:
# =====================
# 6. INFERENCE (BEAM SEARCH)
# =====================
def generate_report_beam(image, encoder, decoder, word2idx, idx2word, beam_width=10, max_len=100):
    encoder.eval()
    decoder.eval()
    with torch.no_grad():
        image = image.unsqueeze(0).to(device)  # [1, 3, 224, 224]
        features = encoder(image)  # [1, 512]
        sequences = [[word2idx["<SOS>"]]]
        scores = torch.zeros(1, device=device)

        for _ in range(max_len):
            all_candidates = []
            for i, seq in enumerate(sequences):
                if seq[-1] == word2idx["<EOS>"]:
                    all_candidates.append((seq, scores[i]))
                    continue
                input_seq = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)
                embedded = decoder.embedding(input_seq)
                h0 = decoder.init_h(features).unsqueeze(0)
                c0 = decoder.init_c(features).unsqueeze(0)
                output, _ = decoder.lstm(embedded, (h0, c0))
                logits = decoder.fc(output[:, -1, :])
                log_probs = torch.log_softmax(logits, dim=-1)
                topk_log_probs, topk_idx = torch.topk(log_probs, beam_width)
                for j in range(beam_width):
                    new_seq = seq + [topk_idx[0, j].item()]
                    new_score = scores[i] + topk_log_probs[0, j]
                    all_candidates.append((new_seq, new_score))
            ordered = sorted(all_candidates, key=lambda x: x[1] / len(x[0]), reverse=True)
            sequences, scores = zip(*ordered[:beam_width])
            scores = torch.tensor(scores)
            if all(seq[-1] == word2idx["<EOS>"] for seq in sequences):
                break
        best_seq = sequences[0]
        return " ".join(idx2word[idx] for idx in best_seq if idx not in [word2idx["<PAD>"], word2idx["<SOS>"], word2idx["<EOS>"]])


In [ ]:
# =====================
# 7. EVALUATION
# =====================
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge

def decode_report(indices):
    return " ".join(idx2word.get(i, "<UNK>") for i in indices if i not in [word2idx["<PAD>"]])

def evaluate_bleu_rouge(gts, preds):
    smoothie = SmoothingFunction().method1
    bleu_scores = []
    rouge = Rouge()
    rouge_scores = []
    for ref, hyp in zip(gts, preds):
        ref = ref.lower().split()
        hyp = hyp.lower().split()
        bleu = sentence_bleu([ref], hyp, smoothing_function=smoothie)
        bleu_scores.append(bleu)
        rouge_scores.append(rouge.get_scores(" ".join(hyp), " ".join(ref))[0])
    avg_bleu = sum(bleu_scores) / len(bleu_scores)
    avg_rouge = {
        metric: sum([s[metric]["f"] for s in rouge_scores]) / len(rouge_scores)
        for metric in ["rouge-1", "rouge-2", "rouge-l"]
    }
    return avg_bleu, avg_rouge

# Run inference on validation set
ground_truth_reports = []
generated_reports = []

for i in range(len(valid_dataset)):
    image, report_tensor = valid_dataset[i]
    if image.dim() != 3:
        print(f"⚠️ Skipping invalid image at index {i}")
        continue
    ground_truth = decode_report(report_tensor.tolist())
    generated = generate_report_beam(image, encoder, decoder, word2idx, idx2word)
    ground_truth_reports.append(ground_truth)
    generated_reports.append(generated)

# Evaluate
bleu, rouge = evaluate_bleu_rouge(ground_truth_reports, generated_reports)
print(f"\n🔍 BLEU: {bleu:.4f}")
print(f"🔴 ROUGE-1 F1: {rouge['rouge-1']:.4f}")
print(f"🟠 ROUGE-2 F1: {rouge['rouge-2']:.4f}")
print(f"🟢 ROUGE-L F1: {rouge['rouge-l']:.4f}")


🔍 BLEU: 0.302463
🔴 ROUGE-1 F1: 0.422342
🟠 ROUGE-2 F1: 0.380348
🟢 ROUGE-L F1: 0.4024786


In [38]:
# Number of reports to preview
N = 10

print("\n📄 Preview of Generated vs. Ground Truth Reports:\n")
for i in range(min(N, len(ground_truth_reports))):
    print(f"🩺 Sample {i+1}")
    print(f"🔹 Ground Truth : {ground_truth_reports[i]}")
    print(f"🔸 Generated    : {generated_reports[i]}")
    print("-" * 80)



📄 Preview of Generated vs. Ground Truth Reports:

🩺 Sample 1
🔹 Ground Truth : <SOS> no information heart size within normal limits stable mediastinal contours densities in the lingula may be compatible with scarring or subsegmental atelectasis scattered chronic appearing irregular interstitial markings no focal alveolar consolidation no definite pleural effusion seen mild bronchovascular crowding without typical findings of pulmonary edema <EOS>
🔸 Generated    : no information heart size within normal limits stable mediastinal contours densities in the lingula may be compatible with scarring or subsegmental atelectasis scattered chronic appearing irregular interstitial markings no focal alveolar consolidation no definite pleural effusion seen mild bronchovascular crowding without typical findings of pulmonary edema
--------------------------------------------------------------------------------
🩺 Sample 2
🔹 Ground Truth : <SOS> lungs are clear bilaterally cardiac and mediastinal silho

In [41]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F

class CNN_Encoder(nn.Module):
    def __init__(self, encoded_dim=512):
        super(CNN_Encoder, self).__init__()
        resnet = models.resnet50(pretrained=True)
        self.resnet = nn.Sequential(*list(resnet.children())[:-2])  # Keep conv features
        self.adaptive_pool = nn.AdaptiveAvgPool2d((14, 14))  # Spatial map
        self.conv = nn.Conv2d(2048, encoded_dim, kernel_size=1)
        self.bn = nn.BatchNorm2d(encoded_dim)

    def forward(self, images):
        features = self.resnet(images)  # [B, 2048, H, W]
        features = self.adaptive_pool(features)  # [B, 2048, 14, 14]
        features = self.conv(features)  # [B, encoded_dim, 14, 14]
        features = self.bn(features)
        features = features.flatten(2).permute(0, 2, 1)  # [B, 196, encoded_dim]
        return features  # Each patch becomes a "token"

class Attention(nn.Module):
    def __init__(self, encoder_dim, decoder_dim):
        super(Attention, self).__init__()
        self.encoder_att = nn.Linear(encoder_dim, decoder_dim)
        self.decoder_att = nn.Linear(decoder_dim, decoder_dim)
        self.full_att = nn.Linear(decoder_dim, 1)

    def forward(self, encoder_out, decoder_hidden):
        att1 = self.encoder_att(encoder_out)  # [B, num_pixels, dec_dim]
        att2 = self.decoder_att(decoder_hidden).unsqueeze(1)  # [B, 1, dec_dim]
        att = self.full_att(torch.tanh(att1 + att2))  # [B, num_pixels, 1]
        alpha = F.softmax(att, dim=1)  # [B, num_pixels, 1]
        context = (encoder_out * alpha).sum(dim=1)  # [B, enc_dim]
        return context, alpha

class LSTM_Decoder_Attention(nn.Module):
    def __init__(self, encoded_dim, hidden_dim, vocab_size, dropout=0.5):
        super(LSTM_Decoder_Attention, self).__init__()
        self.attention = Attention(encoded_dim, hidden_dim)
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTMCell(hidden_dim + encoded_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        self.init_h = nn.Linear(encoded_dim, hidden_dim)
        self.init_c = nn.Linear(encoded_dim, hidden_dim)

    def forward(self, encoder_out, captions):
        batch_size = encoder_out.size(0)
        num_steps = captions.size(1)

        embeddings = self.dropout(self.embedding(captions))  # [B, T, H]
        h, c = self.init_hidden_state(encoder_out.mean(dim=1))  # [B, H]

        outputs = []
        for t in range(num_steps):
            context, _ = self.attention(encoder_out, h)  # [B, enc_dim]
            lstm_input = torch.cat([embeddings[:, t], context], dim=1)
            h, c = self.lstm(lstm_input, (h, c))
            output = self.fc(h)
            outputs.append(output.unsqueeze(1))
        return torch.cat(outputs, dim=1)  # [B, T, Vocab]

    def init_hidden_state(self, encoder_mean):
        h = self.init_h(encoder_mean)
        c = self.init_c(encoder_mean)
        return h, c


In [43]:
import time
from tqdm import tqdm

# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab_size = len(word2idx)

encoder = CNN_Encoder(encoded_dim=512).to(device)
decoder = LSTM_Decoder_Attention(encoded_dim=512, hidden_dim=512, vocab_size=vocab_size).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=word2idx["<PAD>"])
optimizer = torch.optim.AdamW(
    list(encoder.parameters()) + list(decoder.parameters()), 
    lr=1e-4, weight_decay=1e-5  # ✅ added weight decay for regularization
)

# Optional: learning rate scheduler
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.8)

# Optional: Gradient Clipping
clip_value = 1.0

# Training Loop
best_loss = float('inf')
patience = 10
patience_counter = 0

for epoch in range(1, 250):
    encoder.train()
    decoder.train()
    total_loss = 0.0
    start_time = time.time()

    # Progress bar
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}", leave=False)

    for images, reports in pbar:
        images, reports = images.to(device), reports.to(device)
        optimizer.zero_grad()

        # Forward pass
        features = encoder(images)
        outputs = decoder(features, reports[:, :-1])

        # Compute loss
        loss = criterion(outputs.view(-1, vocab_size), reports[:, 1:].reshape(-1))
        total_loss += loss.item()

        # Backward pass
        loss.backward()

        # ✅ Gradient Clipping
        nn.utils.clip_grad_norm_(decoder.parameters(), clip_value)
        nn.utils.clip_grad_norm_(encoder.parameters(), clip_value)

        optimizer.step()

        # Update progress bar
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

    scheduler.step()  # ✅ Adjust LR
    avg_loss = total_loss / len(train_loader)
    elapsed = time.time() - start_time

    print(f"✅ Epoch {epoch} | Loss: {avg_loss:.4f} | Time: {elapsed:.1f}s")

    # ✅ Save best model (optional)
    if avg_loss < best_loss:
        best_loss = avg_loss
        patience_counter = 0
        torch.save(encoder.state_dict(), "best_encoder.pth")
        torch.save(decoder.state_dict(), "best_decoder.pth")
        print("🔸 Best model saved.")
    else:
        patience_counter += 1

    # ✅ Early stopping
    if patience_counter >= patience:
        print("⏹️ Early stopping triggered.")
        break


✅ Epoch 1 | Loss: 5.7339 | Time: 133.4s
🔸 Best model saved.


✅ Epoch 2 | Loss: 4.8659 | Time: 130.8s
🔸 Best model saved.


✅ Epoch 3 | Loss: 4.6152 | Time: 128.3s
🔸 Best model saved.


✅ Epoch 4 | Loss: 4.4022 | Time: 130.1s
🔸 Best model saved.


✅ Epoch 5 | Loss: 4.1661 | Time: 128.0s
🔸 Best model saved.


...

✅ Epoch 245 | Loss: 0.1471 | Time: 129.8s
🔸 Best model saved.


✅ Epoch 246 | Loss: 0.1475 | Time: 130.2s


✅ Epoch 247 | Loss: 0.1477 | Time: 143.0s


✅ Epoch 248 | Loss: 0.1464 | Time: 195.3s
🔸 Best model saved.


✅ Epoch 249 | Loss: 0.1455 | Time: 191.7s
🔸 Best model saved.


In [ ]:
# Load the best model
encoder.load_state_dict(torch.load("best_encoder.pth"))
decoder.load_state_dict(torch.load("best_decoder.pth"))
encoder.eval()
decoder.eval()


In [49]:
def generate_report_beam(image, encoder, decoder, word2idx, idx2word, beam_width=5, max_len=100):
    with torch.no_grad():
        image = image.unsqueeze(0).to(device)  # [1, 3, 224, 224]
        features = encoder(image)              # [1, num_patches, enc_dim]
        sequences = [[word2idx["<SOS>"]]]
        scores = torch.zeros(1, device=device)

        h, c = decoder.init_hidden_state(features.mean(dim=1))

        for _ in range(max_len):
            all_candidates = []
            for i, seq in enumerate(sequences):
                if seq[-1] == word2idx["<EOS>"]:
                    all_candidates.append((seq, scores[i]))
                    continue

                input_seq = torch.tensor(seq, dtype=torch.long, device=device).unsqueeze(0)
                embedding = decoder.embedding(input_seq[:, -1])
                context, _ = decoder.attention(features, h)
                lstm_input = torch.cat([embedding, context], dim=1)
                h, c = decoder.lstm(lstm_input, (h, c))
                output = decoder.fc(h)
                log_probs = torch.log_softmax(output, dim=-1)
                topk_log_probs, topk_idx = torch.topk(log_probs, beam_width)

                for j in range(beam_width):
                    new_seq = seq + [topk_idx[0, j].item()]
                    new_score = scores[i] + topk_log_probs[0, j]
                    all_candidates.append((new_seq, new_score))

            ordered = sorted(all_candidates, key=lambda x: x[1] / len(x[0]), reverse=True)
            sequences, scores = zip(*ordered[:beam_width])
            scores = torch.tensor(scores)

            if all(seq[-1] == word2idx["<EOS>"] for seq in sequences):
                break

        best_seq = sequences[0]
        return " ".join(idx2word[idx] for idx in best_seq if idx not in [word2idx["<PAD>"], word2idx["<SOS>"], word2idx["<EOS>"]])


In [ ]:
ground_truth_reports = []
generated_reports = []

for i in range(len(valid_dataset)):
    image, report_tensor = valid_dataset[i]
    if image.dim() != 3:
        continue
    ground_truth = " ".join([idx2word.get(idx, "<UNK>") for idx in report_tensor.tolist() if idx not in [word2idx["<PAD>"]]])
    generated = generate_report_beam(image, encoder, decoder, word2idx, idx2word)

    ground_truth_reports.append(ground_truth)
    generated_reports.append(generated)


In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge import Rouge

# Function to evaluate BLEU and ROUGE
def evaluate_bleu_rouge(gts, preds):
    smoothie = SmoothingFunction().method1
    bleu_scores = []
    rouge = Rouge()
    rouge_scores = []

    for ref, hyp in zip(gts, preds):
        ref = ref.lower().split()
        hyp = hyp.lower().split()
        bleu = sentence_bleu([ref], hyp, smoothing_function=smoothie)
        bleu_scores.append(bleu)
        rouge_scores.append(rouge.get_scores(" ".join(hyp), " ".join(ref))[0])

    avg_bleu = sum(bleu_scores) / len(bleu_scores)
    avg_rouge = {
        metric: sum([s[metric]["f"] for s in rouge_scores]) / len(rouge_scores)
        for metric in ["rouge-1", "rouge-2", "rouge-l"]
    }
    return avg_bleu, avg_rouge, bleu_scores

# Lists to hold reports
ground_truth_reports = []
generated_reports = []

# Generate reports
for i in range(len(valid_dataset)):
    image, report_tensor = valid_dataset[i]
    if image.dim() != 3:
        continue
    ground_truth = " ".join([idx2word.get(idx, "<UNK>") for idx in report_tensor.tolist() if idx not in [word2idx["<PAD>"], word2idx["<SOS>"], word2idx["<EOS>"]]])
    generated = generate_report_beam(image, encoder, decoder, word2idx, idx2word)

    ground_truth_reports.append(ground_truth)
    generated_reports.append(generated)

# Evaluate
bleu, rouge, bleu_scores = evaluate_bleu_rouge(ground_truth_reports, generated_reports)

# Print average scores
print(f"\n🔍 Average BLEU: {bleu:.4f}")
print(f"🔴 ROUGE-1 F1: {rouge['rouge-1']:.4f}")
print(f"🟠 ROUGE-2 F1: {rouge['rouge-2']:.4f}")
print(f"🟢 ROUGE-L F1: {rouge['rouge-l']:.4f}")




0.25321
🔴 ROUGE-1 F1: 0.402564
🟠 ROUGE-2 F1: 0.293214
🟢 ROUGE-L F1: 0.3702568
